### Databricks Lakeflow Jobs — Complete Notes

#### 1. Overview

**Lakeflow Jobs** is Databricks' native **orchestration** framework — used to build, schedule, and monitor workflows (DAGs) directly inside Databricks, reducing dependency on third-party orchestrators (ADF, Airflow, etc.).

- Recently rebranded/expanded as part of Databricks' **Lakeflow** family: Lakeflow Jobs, Lakeflow Pipelines (formerly Delta Live Tables / DLT), Lakeflow Designer.
- Moved from "public preview" to **Generally Available (GA)** in 2025 — actively growing in adoption.
- Found under **Jobs & Pipelines** in the workspace left nav (not just under Data Engineering — also used by Data Scientists/AI teams).

#### Why it matters
- Currently most orgs orchestrate Databricks notebooks via **Azure Data Factory** or **Apache Airflow**.
- As Databricks adoption grows, Lakeflow Jobs reduces the need for external orchestration tools and makes Databricks equally powerful across any cloud.
- A Data Engineer who knows Lakeflow Jobs can orchestrate work for Data Scientists/AI teams too — not just ETL.

---

#### 2. Core Concepts

##### Orchestration vs DAG
- **Orchestration** = coordinating a flow of work (borrowed from music — "orchestrating" different parts to run in the right order).
- **DAG (Directed Acyclic Graph)** = more technical term for the same idea — a graph of steps (nodes) with defined direction, and (typically) **no cycles** (it doesn't loop back to a prior step within a single run — each scheduled run starts a fresh new DAG instance instead).

##### Jobs vs Pipelines (Databricks-specific distinction)
| | Pipelines (Lakeflow Pipelines / DLT) | Jobs (Lakeflow Jobs) |
|---|---|---|
| Purpose | **Processes data** (bronze/silver/gold ETL logic) | **Orchestrates work** (notebooks, pipelines, SQL, other jobs, etc.) |
| Scope | One dedicated data-processing flow | Can orchestrate **multiple pipelines**, notebooks, SQL, conditionals, loops, other jobs |
| Relationship | A pipeline is like **one unit of work** | A job can run **many pipelines/tasks** together, with dependencies |

> Key insight: You can orchestrate pipelines *within* jobs (e.g., run bronze pipeline → silver pipeline → gold pipeline in sequence), plus add conditionals, loops, and control flow that pipelines alone can't do.

---

#### 3. Tasks — The Building Block

A **Job** = a container that holds one or more **Tasks**. Each task is a unit of work (a notebook, SQL query/file, Python script, dbt job, another job, pipeline refresh, dashboard refresh, alert, JAR, Power BI, etc.).

##### Basic patterns
- **Single task**: one notebook runs on a schedule.
- **Sequential tasks**: Task B `depends on` Task A → runs only after A succeeds.
- **Parallel tasks**: Task B and Task C both `depend on` Task A → both run together once A finishes.

##### Task configuration options
- **Task name** — any label.
- **Type** — Notebook, Python script, Python wheel, SQL (query/file/pipeline/alert), dbt task, Run Job (nested job), Pipeline refresh, Dashboard refresh, JAR, Power BI, For Each, If/Else, Clean Room task.
- **Source** — Workspace (or Git).
- **Path** — location of the notebook/file.
- **Compute** — cluster to run on (serverless recommended for simplicity).
- **Parameters** — key/value inputs to the task.
- **Notifications** — email/webhook alerts on task start/success/failure (configurable **per task**, unlike ADF where this needs Logic Apps).
- **Retry policy** — e.g., retry 1 time, wait 15 seconds before retry (handles transient/networking failures without failing the whole pipeline).
- **Depends on** — defines the upstream task(s).
- **Run if dependencies** — controls trigger condition: *all succeeded* (default), *all failed*, *at least one failed*, etc.

---

#### 4. Control Flow

##### If/Else Conditions
- Add a task of type **"If/Else condition"**.
- Define a condition using **dynamic value references**, e.g.:
  ```
  {{job.start_time.iso_weekday}} == 7
  ```
- Common functions: `job.start_time.is_weekday`, `job.start_time.iso_weekday`, `job.start_time.day`.
- Attach downstream tasks to the **true** branch or **false** branch.
- Real use case: run Task B only on weekends, Task C only on weekdays.

##### Loops (For Each)
- Add a task of type **"For Each"**.
- Requires an **iterable input** — typically an **array** (list). (Tuples/strings are iterable too, but arrays are used ~99% of the time; arrays can contain dictionaries.)
- Input can be:
  - A **hardcoded array**, e.g. `[1,2,3,4,5]`
  - A **dynamic array** from a previous task's output (notebook or SQL)
- **Concurrency** (optional) — how many iterations run in parallel (default = 1, sequential). Recommended range: **10–20** max, not "as high as possible."
- Inside the loop, attach a task (e.g., a notebook) that becomes the "looped task" — it runs once per array element.
- Each iteration is tracked (iteration 0, 1, 2 … n-1) and viewable individually in run history.

---

#### 5. Passing Data Between Tasks (Task Values)

This is the **core mechanic** for building dynamic, parameterized pipelines.

##### Setting a value (in a notebook)
```python
dbutils.jobs.taskValues.set(key="total_records", value=total_records)
```
- `key` = identifier you'll reference later.
- `value` = the data to pass (can be any serializable type — number, string, list, dict).

##### Getting a value — Option A: dynamic value reference in job UI
Used when passing a value into a **task's parameter field** (via UI, not code):
```
{{tasks.<task_name>.values.<key>}}
```
Example: `{{tasks.task_x.values.total_records}}`

##### Getting a value — Option B: inline within another notebook's code
```python
dbutils.jobs.taskValues.get(taskKey="task_x", key="total_records")
```

##### Widgets / Parameters (in a notebook)
```python
dbutils.widgets.text("par1", "")       # create parameter/widget
dbutils.widgets.get("par1")            # read parameter value
```
- Parameters passed via the job UI always arrive as **strings** — use `eval()` or manual parsing to convert to list/int/etc. if needed.

##### SQL files vs SQL queries
- **SQL Files** (not SQL Queries) support **dynamic parameters** in Lakeflow Jobs — SQL Queries only accept static hardcoded values.
- Define a SQL parameter using `:variable_name` syntax:
  ```sql
  SELECT * FROM db_jobs.default.orders WHERE id = :order_id
  ```
- In the job task config, set the parameter dynamically:
  ```
  {{tasks.task_x.values.order_id}}
  ```
- To reference SQL task **output** in another notebook:
  ```
  {{tasks.<sql_task_name>.output.rows}}       # all rows → list of dicts
  {{tasks.<sql_task_name>.output.first_row}}  # first row only
  ```

---

#### 6. Real-World Pattern: Dynamic File Ingestion (For Each + Array of Dicts)

Common production pattern for ingesting N files without duplicating code:

1. **Source**: multiple files (file1, file2, file3 … or 300 files).
2. **Array-of-dictionaries task**: a notebook (or SQL query) returns:
   ```python
   file_names = [
       {"file_name": "orders"},
       {"file_name": "products"},
       {"file_name": "regions"}
   ]
   dbutils.jobs.taskValues.set(key="file_names", value=file_names)
   ```
3. **For Each task**: input = `{{tasks.array.values.file_names}}`
4. **Looped notebook task** (e.g., ingestion notebook): parameter `file_name` = `{{input.file_name}}`
   - Inside `input` refers to the current loop element (like `i` in a Python `for i in array` loop), and `.file_name` accesses that dict key.
5. Ingestion notebook reads a parameterized path:
   ```python
   dbutils.widgets.text("file_name", "")
   file_name = dbutils.widgets.get("file_name")
   df = spark.read.format("parquet").load(f".../raw_data/{file_name}.parquet")
   df.write.format("delta").mode("overwrite").save(f".../sink/{file_name}")
   ```

##### Best practice upgrade: Mapping Table instead of hardcoded array in a notebook
- Hardcoding the array inside a notebook is hard to maintain (no easy DML, not scalable for large lists).
- **Recommended**: store the file list in a **Delta table** (mapping table) and read it via a **SQL query task**:
  ```sql
  CREATE TABLE db_jobs.default.mapping (file_name STRING);
  INSERT INTO db_jobs.default.mapping VALUES ('orders'), ('products'), ('regions');
  ```
  ```sql
  SELECT * FROM db_jobs.default.mapping
  ```
- For Each input becomes: `{{tasks.<sql_task_name>.output.rows}}`
- Easier to update/maintain (DML), better for large lists, more "professional"/production-friendly than a notebook array.

---

#### 7. Alerts

- Found under **Alerts** (left nav) → **Create Alert** (use modern alerts, not legacy — legacy is being deprecated).
- Define a SQL query that returns a single evaluable value, e.g.:
  ```sql
  SELECT SUM(total_amount) AS total_revenue FROM db_jobs.default.orders
  ```
- Set a **condition**, e.g. `total_revenue >= 300`.
- Set a **notification destination** (email) — configured under **Settings → Developer/Notifications → Manage → Add destination**.
- Can be scheduled independently (daily/weekly/monthly) and can also be embedded as a **task type inside a Lakeflow Job**.
- Use cases: alert on business KPIs (e.g., revenue threshold crossed), not just pipeline failures.

---

#### 8. Scheduling & Triggers

Configured under **Job → Schedules & Triggers → Add Trigger**.

| Trigger Type | Description |
|---|---|
| **Scheduled** | Cron-based; set specific time/frequency (e.g., daily at 9:30 AM). UI supports simple and cron/advanced syntax. |
| **File arrival** | Triggers when a new file lands in a specified storage location — equivalent to **Storage Event Triggers** in Azure Data Factory. Requires cloud storage integration/permissions. |
| **Continuous** | Job keeps re-triggering a new run as soon as the previous one finishes (similar concept to Spark Structured Streaming continuous mode). Currently less common; expected to grow. |

---

#### 9. Job-Level Settings

- **Job Parameters**: parameters defined at the **parent job level** (rather than per-task), usable across multiple tasks — useful when several tasks need the same input.
- **Compute**:
  - **Serverless** (recommended, used throughout the tutorial) — no cluster management.
  - **Job compute** — dedicated cluster that spins up for the job and terminates automatically after.
- **Job Notifications**: configure email alerts at the **job level** for on start / on success / on failure / on duration threshold exceeded — much simpler than Azure Data Factory (which requires Logic Apps + Web Activity for equivalent behavior).
- **Repair Run**: if a job fails partway, use **"Repair Task"** to re-run *only the failed task(s) onward* instead of the entire pipeline from scratch — saves time/compute.
- **Performance-optimized toggle**: enable this on task/job settings to reduce cluster startup latency during testing (otherwise runs take much longer).

---

#### 10. Other Task Types (Brief)

| Task Type | Purpose |
|---|---|
| Notebook | Run a Databricks notebook |
| Python script / Python wheel | Run standalone Python code/packages |
| SQL query / SQL file / SQL pipeline / SQL alert | Run SQL logic, refresh pipelines, or trigger alerts |
| dbt task | Run a dbt job/model |
| Run Job | Nest another job inside this job (job-within-job orchestration) |
| Pipeline refresh | Trigger a Lakeflow Pipeline (DLT) run |
| Dashboard refresh | Refresh a Databricks dashboard's underlying dataset |
| JAR | Run compiled Java/Scala code |
| Power BI | Refresh/integrate with Power BI |
| Clean Room task | Secure, cross-organization collaboration on shared data/notebooks without exposing raw data directly (advanced/enterprise use case, typically not hands-on for a data engineer) |
| For Each | Loop a task over an iterable array |
| If/Else condition | Branch logic based on a dynamic condition |

---

#### 11. Quick-Reference Cheat Sheet (Interview Recall)

- **Job** = orchestration container.
- **Task** = unit of work inside a job-
- **Pipeline** = data-processing unit (can be run *inside* a job).
- **DAG** = directed acyclic graph; each scheduled run = a fresh DAG instance (not cyclic within one run).
- Task dependency control: `depends on` + `run if` (all succeeded / all failed / at least one failed, etc.).
- Dynamic values: `dbutils.jobs.taskValues.set(key, value)` → `{{tasks.<task>.values.<key>}}` or `dbutils.jobs.taskValues.get(taskKey, key)`.
- SQL output reference: `{{tasks.<task>.output.rows}}` / `{{tasks.<task>.output.first_row}}`.
- Loops need an **iterable** (array of values or array of dicts); reference current item as `{{input.<key>}}`.
- Use **mapping tables** (Delta tables) instead of hardcoded notebook arrays for production-scale file lists.
- **Repair Task** = re-run only failed tasks, not the whole job — a strong practical/interview talking point about pipeline reliability.
- Retry policy + notifications + alerts = built-in reliability/monitoring features that reduce dependency on ADF + Logic Apps.
- Triggers: Scheduled (cron), File Arrival (storage event), Continuous (streaming-like re-trigger).